In [2]:
import sys
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "mlflow"], check=True)

CompletedProcess(args=['c:\\Users\\tayla\\AppData\\Local\\Python\\pythoncore-3.14-64\\python.exe', '-m', 'pip', 'install', 'mlflow'], returncode=0)

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder, FunctionTransformer
from sklearn.feature_extraction import FeatureHasher
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score,
    ConfusionMatrixDisplay,
    classification_report,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)

import joblib
import mlflow
import mlflow.sklearn

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

RANDOM_STATE = 42

In [ ]:
# ── Load raw data ─────────────────────────────────────────────────────────────
RAW_PATH = Path("../data/raw/airlines_delay.csv")

if not RAW_PATH.exists():
    raise FileNotFoundError(f"Raw dataset not found at {RAW_PATH}.")

df = pd.read_csv(RAW_PATH)

print("Source :", RAW_PATH)
print("Shape  :", df.shape)
display(df.head())

FileNotFoundError: Raw dataset not found at ..\data\raw\airlines_delay.csv.
Please place airlines_delay.csv in data/raw/

In [ ]:
# ── Feature Engineering ───────────────────────────────────────────────────────

# Time (minutes since midnight) → hour of day
df["DepartureHour"] = df["Time"] // 60

# Route: combines origin + destination (captures route-level delay patterns)
df["Route"] = df["AirportFrom"] + "-" + df["AirportTo"]

# AirlineRoute: merges Airline + AirportFrom + AirportTo into one string (3 columns → 1)
# e.g. 'WN_SFO_LAX' — captures carrier-specific route delay patterns
df["AirlineRoute"] = df["Airline"] + "_" + df["AirportFrom"] + "_" + df["AirportTo"]

DROP_COLS = ["id", "Flight", "Time"]
df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

print("Columns after feature engineering:", df.columns.tolist())
print("Shape:", df.shape)
display(df.head())

In [ ]:
# ── Target + Train/Test Split ─────────────────────────────────────────────────
TARGET = "Delay"

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Train: {X_train.shape[0]:,} rows | Test: {X_test.shape[0]:,} rows")
print("Class distribution (test):")
display(y_test.value_counts(normalize=True).mul(100).round(1))

In [ ]:
# ── Helper: evaluate a fitted pipeline ───────────────────────────────────────
def evaluate(pipeline, X_test, y_test, label="Model"):
    y_pred  = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    metrics = {
        "Accuracy" : accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall"   : recall_score(y_test, y_pred, zero_division=0),
        "F1"       : f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC"  : roc_auc_score(y_test, y_proba),
    }

    print(f"\n{'='*55}")
    print(f"  {label}")
    print(f"{'='*55}")
    for k, v in metrics.items():
        print(f"  {k:<12}: {v:.4f}")
    print()
    print(classification_report(y_test, y_pred, target_names=["No Delay", "Delay"]))

    return metrics, y_pred, y_proba

---
## Strategy A — Target Encoding (same as Random Forest)

In [ ]:
# Features for Strategy A
TARGET_ENC_A = ["Airline", "AirportFrom", "AirportTo", "Route"]
NUMERIC_A    = ["DayOfWeek", "Length", "DepartureHour"]
FEATURES_A   = TARGET_ENC_A + NUMERIC_A

preprocessor_A = ColumnTransformer(
    transformers=[
        ("target_enc", TargetEncoder(target_type="binary", random_state=RANDOM_STATE), TARGET_ENC_A),
        ("passthrough", "passthrough", NUMERIC_A),
    ],
    remainder="drop",
)

# Depth grid search — find the sweet spot between underfitting and overfitting
depths_A = [3, 5, 8, 12, 20, None]  # None = unlimited
results_A = {}

for depth in depths_A:
    pipe = Pipeline([
        ("preprocessor", preprocessor_A),
        ("classifier", DecisionTreeClassifier(
            max_depth=depth,
            min_samples_leaf=50,
            random_state=RANDOM_STATE,
        )),
    ])
    pipe.fit(X_train[FEATURES_A], y_train)
    metrics, _, _ = evaluate(pipe, X_test[FEATURES_A], y_test,
                             label=f"DT (Target Enc) | depth={depth}")
    results_A[depth] = {"pipeline": pipe, "metrics": metrics}

In [ ]:
# Best depth for Strategy A (by F1)
best_depth_A = max(results_A, key=lambda d: results_A[d]["metrics"]["F1"])
best_pipe_A  = results_A[best_depth_A]["pipeline"]
best_metrics_A = results_A[best_depth_A]["metrics"]

print(f"Best depth (Strategy A): {best_depth_A}")
print(f"F1: {best_metrics_A['F1']:.4f} | ROC-AUC: {best_metrics_A['ROC-AUC']:.4f}")

---
## Strategy B — Combined Feature: 3 Columns → 1 (AirlineRoute)

In [ ]:
# AirlineRoute combines Airline + AirportFrom + AirportTo into one string
# e.g. 'WN_SFO_LAX' — then Target-encoded as its delay rate
# This captures carrier-specific route patterns with a single feature instead of three

TARGET_ENC_B = ["AirlineRoute", "Route"]
NUMERIC_B    = ["DayOfWeek", "Length", "DepartureHour"]
FEATURES_B   = TARGET_ENC_B + NUMERIC_B

preprocessor_B = ColumnTransformer(
    transformers=[
        ("target_enc", TargetEncoder(target_type="binary", random_state=RANDOM_STATE), TARGET_ENC_B),
        ("passthrough", "passthrough", NUMERIC_B),
    ],
    remainder="drop",
)

depths_B  = [3, 5, 8, 12, 20, None]
results_B = {}

for depth in depths_B:
    pipe = Pipeline([
        ("preprocessor", preprocessor_B),
        ("classifier", DecisionTreeClassifier(
            max_depth=depth,
            min_samples_leaf=50,
            random_state=RANDOM_STATE,
        )),
    ])
    pipe.fit(X_train[FEATURES_B], y_train)
    metrics, _, _ = evaluate(pipe, X_test[FEATURES_B], y_test,
                             label=f"DT (AirlineRoute 3→1) | depth={depth}")
    results_B[depth] = {"pipeline": pipe, "metrics": metrics}

In [ ]:
# Best depth for Strategy B (by F1)
best_depth_B   = max(results_B, key=lambda d: results_B[d]["metrics"]["F1"])
best_pipe_B    = results_B[best_depth_B]["pipeline"]
best_metrics_B = results_B[best_depth_B]["metrics"]

print(f"Best depth (Strategy B): {best_depth_B}")
print(f"F1: {best_metrics_B['F1']:.4f} | ROC-AUC: {best_metrics_B['ROC-AUC']:.4f}")

---
## Strategy C — Feature Hashing

In [ ]:
# Feature Hashing: maps each category string to a fixed-size numeric vector
# via a hash function — no training needed, no new columns per unseen category.
# n_features=64: output vector size (power of 2, tune if needed)

CAT_COLS_C = ["Airline", "AirportFrom", "AirportTo", "AirlineRoute"]
NUMERIC_C  = ["DayOfWeek", "Length", "DepartureHour"]

def hash_encode(X_df, cat_cols=CAT_COLS_C, n_features=64):
    """Convert categorical columns to a hashed feature matrix and concatenate with numeric cols."""
    hasher = FeatureHasher(n_features=n_features, input_type="string")

    # Each row: list of 'colname=value' strings
    cat_data = X_df[cat_cols].astype(str).apply(
        lambda row: [f"{col}={val}" for col, val in zip(cat_cols, row)],
        axis=1,
    ).tolist()

    hashed = hasher.transform(cat_data).toarray()
    numeric = X_df[NUMERIC_C].values

    return np.hstack([hashed, numeric])

# Build matrices directly (FeatureHasher doesn't fit into ColumnTransformer easily)
X_train_C = hash_encode(X_train)
X_test_C  = hash_encode(X_test)

print(f"Hashed feature matrix shape: {X_train_C.shape}")
print(f"  ({64} hash buckets + {len(NUMERIC_C)} numeric features)")

In [ ]:
depths_C  = [3, 5, 8, 12, 20, None]
results_C = {}

for depth in depths_C:
    clf = DecisionTreeClassifier(
        max_depth=depth,
        min_samples_leaf=50,
        random_state=RANDOM_STATE,
    )
    clf.fit(X_train_C, y_train)

    y_pred  = clf.predict(X_test_C)
    y_proba = clf.predict_proba(X_test_C)[:, 1]

    metrics = {
        "Accuracy" : accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall"   : recall_score(y_test, y_pred, zero_division=0),
        "F1"       : f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC"  : roc_auc_score(y_test, y_proba),
    }

    print(f"DT (Hash Encoding) | depth={depth} → F1: {metrics['F1']:.4f} | AUC: {metrics['ROC-AUC']:.4f}")
    results_C[depth] = {"clf": clf, "metrics": metrics}

In [ ]:
# Best depth for Strategy C (by F1)
best_depth_C   = max(results_C, key=lambda d: results_C[d]["metrics"]["F1"])
best_clf_C     = results_C[best_depth_C]["clf"]
best_metrics_C = results_C[best_depth_C]["metrics"]

print(f"Best depth (Strategy C): {best_depth_C}")
print(f"F1: {best_metrics_C['F1']:.4f} | ROC-AUC: {best_metrics_C['ROC-AUC']:.4f}")

---
## Best Model: Plots + Evaluation

In [ ]:
# Pick the overall best strategy by F1
all_best = {
    "A_TargetEnc"  : (best_metrics_A, "A"),
    "B_AirlineRoute": (best_metrics_B, "B"),
    "C_HashEnc"    : (best_metrics_C, "C"),
}

winner_name = max(all_best, key=lambda k: all_best[k][0]["F1"])
winner_strategy = all_best[winner_name][1]
print(f"\nBest overall strategy: {winner_name} (Strategy {winner_strategy})")

if winner_strategy == "A":
    best_pipe    = best_pipe_A
    best_metrics = best_metrics_A
    best_depth   = best_depth_A
    X_test_best  = X_test[FEATURES_A]
    y_pred_best  = best_pipe.predict(X_test_best)
    y_proba_best = best_pipe.predict_proba(X_test_best)[:, 1]
    encoding_label = "TargetEncoding"
elif winner_strategy == "B":
    best_pipe    = best_pipe_B
    best_metrics = best_metrics_B
    best_depth   = best_depth_B
    X_test_best  = X_test[FEATURES_B]
    y_pred_best  = best_pipe.predict(X_test_best)
    y_proba_best = best_pipe.predict_proba(X_test_best)[:, 1]
    encoding_label = "AirlineRoute_3to1"
else:
    best_pipe    = best_clf_C
    best_metrics = best_metrics_C
    best_depth   = best_depth_C
    y_pred_best  = best_clf_C.predict(X_test_C)
    y_proba_best = best_clf_C.predict_proba(X_test_C)[:, 1]
    encoding_label = "FeatureHashing"

In [ ]:
# Confusion Matrix
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_best,
    display_labels=["No Delay", "Delay"],
    cmap="Blues",
    ax=ax,
)
ax.set_title(f"Confusion Matrix — Decision Tree ({winner_name})")
plt.tight_layout()
plt.show()

In [ ]:
# ROC + Precision-Recall
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

RocCurveDisplay.from_predictions(y_test, y_proba_best, ax=ax1, name=f"DT ({winner_name})")
ax1.plot([0, 1], [0, 1], "k--", label="Random (AUC = 0.5)")
ax1.set_title("ROC Curve — Decision Tree")
ax1.legend()

PrecisionRecallDisplay.from_predictions(y_test, y_proba_best, ax=ax2, name=f"DT ({winner_name})")
ax2.axhline(y_test.mean(), color="k", linestyle="--",
            label=f"No-skill baseline ({y_test.mean():.2f})")
ax2.set_title("Precision-Recall Curve — Decision Tree")
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Predicted Probability Distribution
plt.figure(figsize=(8, 4))
plt.hist(y_proba_best[y_test == 0], bins=50, alpha=0.6,
         label="No Delay (actual)", color="steelblue")
plt.hist(y_proba_best[y_test == 1], bins=50, alpha=0.6,
         label="Delay (actual)", color="tomato")
plt.axvline(0.5, color="black", linestyle="--", label="Threshold 0.5")
plt.title(f"Predicted Probability Distribution — Decision Tree ({winner_name})")
plt.xlabel("P(Delay)")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Tree visualization (only useful for shallow depths)
if best_depth is not None and best_depth <= 5:
    clf_step = best_pipe.named_steps["classifier"] if hasattr(best_pipe, 'named_steps') else best_pipe
    plt.figure(figsize=(20, 8))
    plot_tree(
        clf_step,
        max_depth=3,           # show top 3 levels for readability
        filled=True,
        impurity=False,
        proportion=True,
        rounded=True,
        fontsize=9,
    )
    plt.title(f"Decision Tree Structure (top 3 levels) — {winner_name}")
    plt.tight_layout()
    plt.show()
else:
    print(f"Tree too deep (depth={best_depth}) to visualize meaningfully — skipping plot.")

---
## Comparison: All Strategies + Previous Models

In [ ]:
# Depth comparison chart (F1 across all strategies)
depths_labels = [str(d) if d is not None else "None" for d in depths_A]

f1_A = [results_A[d]["metrics"]["F1"] for d in depths_A]
f1_B = [results_B[d]["metrics"]["F1"] for d in depths_B]
f1_C = [results_C[d]["metrics"]["F1"] for d in depths_C]

plt.figure(figsize=(10, 5))
plt.plot(depths_labels, f1_A, marker="o", label="A: Target Encoding")
plt.plot(depths_labels, f1_B, marker="s", label="B: AirlineRoute (3→1)")
plt.plot(depths_labels, f1_C, marker="^", label="C: Feature Hashing")
plt.xlabel("max_depth")
plt.ylabel("F1 Score")
plt.title("Decision Tree F1 by Depth and Encoding Strategy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Load previous model metrics and append Decision Tree results
PROCESSED_DIR = Path("../data/processed")

dt_rows = [
    {"model": f"DecisionTree_TargetEnc_d{best_depth_A}", **best_metrics_A},
    {"model": f"DecisionTree_AirlineRoute_d{best_depth_B}", **best_metrics_B},
    {"model": f"DecisionTree_HashEnc_d{best_depth_C}", **best_metrics_C},
]

prev_files = ["baseline_metrics.csv", "rf_metrics.csv", "xgb_metrics.csv"]
prev_dfs   = []
for fname in prev_files:
    fpath = PROCESSED_DIR / fname
    if fpath.exists():
        prev_dfs.append(pd.read_csv(fpath))
    else:
        print(f"Not found (optional): {fname}")

comparison_df = pd.concat(
    prev_dfs + [pd.DataFrame(dt_rows)],
    ignore_index=True,
)

metric_cols = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
print("\nFull Model Comparison:")
display(comparison_df[["model"] + metric_cols].round(4))

---
## Save Model + Metrics

In [ ]:
# Save best Decision Tree pipeline
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODELS_DIR / "dt_model.pkl"
joblib.dump(best_pipe, MODEL_PATH)
print(f"Model saved   : {MODEL_PATH}")

# Save DT metrics (best strategy only)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
dt_best_row = {"model": f"DecisionTree_{encoding_label}_d{best_depth}", **best_metrics}
dt_metrics_df = pd.DataFrame([dt_best_row])
DT_METRICS_PATH = PROCESSED_DIR / "dt_metrics.csv"
dt_metrics_df.to_csv(DT_METRICS_PATH, index=False)
print(f"Metrics saved : {DT_METRICS_PATH}")

---
## MLflow Logging

In [ ]:
mlflow.set_experiment("DelayPredict")

# Log all three strategies as separate MLflow runs
strategy_configs = [
    ("DecisionTree_TargetEncoding",  "TargetEncoding",  best_depth_A, best_metrics_A, best_pipe_A),
    ("DecisionTree_AirlineRoute3to1","AirlineRoute3to1",best_depth_B, best_metrics_B, best_pipe_B),
    ("DecisionTree_FeatureHashing",  "FeatureHashing",  best_depth_C, best_metrics_C, best_clf_C),
]

for run_name, encoding, depth, metrics, model in strategy_configs:
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({
            "model"          : "DecisionTree",
            "encoding"       : encoding,
            "max_depth"      : str(depth),
            "min_samples_leaf": 50,
            "random_state"   : RANDOM_STATE,
            "train_size"     : X_train.shape[0],
            "test_size"      : X_test.shape[0],
        })
        mlflow.log_metrics({
            "accuracy" : metrics["Accuracy"],
            "precision": metrics["Precision"],
            "recall"   : metrics["Recall"],
            "f1"       : metrics["F1"],
            "roc_auc"  : metrics["ROC-AUC"],
        })
        mlflow.sklearn.log_model(model, artifact_path="model")
        print(f"Logged: {run_name}")

print("\nAll runs logged. Start MLflow UI with: mlflow ui")

In [ ]:
# Summary
print("DECISION TREE RESULTS SUMMARY")
print("=" * 72)
print(f"Train: {X_train.shape[0]:,} rows | Test: {X_test.shape[0]:,} rows")
print(f"Best strategy : {winner_name}")
print(f"Best depth    : {best_depth}")
print(f"Encoding      : {encoding_label}")
print()
print(f"{'Model':<42}" + "".join(f"{m:>10}" for m in metric_cols))
print("-" * 92)
for _, row in comparison_df.iterrows():
    vals = "".join(f"{row[m]:>10.4f}" for m in metric_cols)
    print(f"{row['model']:<42}{vals}")